## 三视角 3D ID 分配（Step1→2→3→4 + 补全 + 视频）

本 notebook 演示：从三视角 tracking 产出的 `*_processed.csv`（允许少量 2D 缺失）出发，
利用校准模型完成 **3D 重建**，并保证：
- **跨帧个体 ID 正确**：Step 4 匈牙利匹配，同一物理个体时间上保持同一 a/b/c；
- **每帧 3 个个体、无缺失值**：Step 1 可选 2D 补全（缺一视角时用另两视角推导），导出前用三次样条补全缺失帧；
- **可选 Step 6**：3D 异常值去除与插值，使轨迹更平滑；
- **生成 2×2 合成视频**：3D 轨迹 + 三视角 2D 叠加。

底层实现位于 `scripts/reconstruction`（参考 Dynamics_of_dominance 的 3D 重建与跨帧追踪）。


**前置条件**：`SET_KEY` 与 04 中处理的片段一致（即 `*_processed.csv` 文件名中的 set key）。  
若 2D 存在少量缺失（某视角某帧不可见），pipeline 会先尝试用另两视角+校准补全 2D，再在 3D 导出后用**缺失帧补全**保证每帧都有 3 个个体坐标。

In [1]:
import sys
from pathlib import Path

# 项目根：从 notebooks 打开则用父目录，否则用当前目录
_cwd = Path.cwd().resolve()
if _cwd.name == "notebooks":
    sys.path.insert(0, str(_cwd.parent))
else:
    sys.path.insert(0, str(_cwd))

# === 路径与参数（按自己环境修改）===
PROCESSED_DIR = r"D:\sleap_output\processed"
SET_KEY = "20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331"
CALIB_MODEL_DIR = r"D:\calibration_model\cali_251217_way1"
OUTPUT_DIR = r"D:\medaka-mating-output\3d_id"
XYZ_BOUNDS = (0.0, 200.0)

PROCESSED_DIR, SET_KEY, CALIB_MODEL_DIR, OUTPUT_DIR

('D:\\sleap_output\\processed',
 '20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331',
 'D:\\calibration_model\\cali_251217_way1',
 'D:\\medaka-mating-output\\3d_id')

In [2]:
# 运行完整 3D ID 分配 pipeline（Step1→2→3→4 + 导出 + 缺失帧补全）
# 保证：跨帧个体 ID 一致（Step 4 匈牙利）、每帧都有 3 个个体（fill_missing_3d 三次样条补全）
# 注意：Step 2 按帧计算 3D 候选，帧数多时需数分钟，会显示「Step2 3D候选」进度条，请勿中断。
from scripts.tracking.multiview_3d_id_assignment import run_multiview_id_pipeline

result = run_multiview_id_pipeline(
    processed_dir=PROCESSED_DIR,
    set_key=SET_KEY,
    model_folder=CALIB_MODEL_DIR,
    output_dir=OUTPUT_DIR,
    xyz_bounds=XYZ_BOUNDS,
    use_fish_id=False,
    save_best_json=True,
    shift_to_positive=False,
    interpolate_3d=True,
    fill_missing_2d=True,
    cross_frame_tracking=True,
    flow_weight=5.0,
    fill_missing_3d=True,
)
n_ok = len(result["best_tracked"])
print(f"完成：共 {n_ok} 帧有 a/b/c 结果；3D CSV 与 best_2d 已写入 {OUTPUT_DIR}")
print("说明：Step1/2/3/4 先用 3D 几何+跨帧追踪尽可能为每帧每条鱼给出 3D 位置，剩余缺帧再由三次样条在时间上补齐（逻辑与 Dynamics_of_dominance 相同：先按几何/配准选点，再用 3D 轨迹本身填洞）。")

Step2 3D候选: 100%|██████████| 929/929 [02:30<00:00,  6.18frame/s]


已保存: D:\medaka-mating-output\3d_id\20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331_3d.csv
已保存: D:\medaka-mating-output\3d_id\20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331_a_3d_interp.csv
已保存: D:\medaka-mating-output\3d_id\20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331_b_3d_interp.csv
完成：共 929 帧有 a/b/c 结果；3D CSV 与 best_2d 已写入 D:\medaka-mating-output\3d_id
说明：Step1/2/3/4 先用 3D 几何+跨帧追踪尽可能为每帧每条鱼给出 3D 位置，剩余缺帧再由三次样条在时间上补齐（逻辑与 Dynamics_of_dominance 相同：先按几何/配准选点，再用 3D 轨迹本身填洞）。


## 分步调试：拆开 Step1 → Step2 → Step3 → Step4 → 导出

下面的单元格把 `run_multiview_id_pipeline` 拆成和 Dynamics_of_dominance 相同顺序的几个小步骤，
每一步都会打印出本步的关键统计，方便检查是哪一步出了问题。

调参 / 排错时，可以只跑这些分步单元；一键版 pipeline（上面的 cell 3）可以先跳过。

In [2]:
# Step 1：从 processed CSV 取 2D 点（可选 2D 补全），形成 per-frame 2D 列表
from collections import Counter

from scripts.reconstruction.calibration import Calibration
from scripts.reconstruction.step1_get_points import (
    load_processed_dfs,
    fill_missing_2d_from_two_views,
    get_points_per_frame,
)

cal = Calibration(CALIB_MODEL_DIR)

dfs_raw = load_processed_dfs(PROCESSED_DIR, SET_KEY)
print("Step1: 原始 processed CSV 加载完成（行数）:")
for k, df in dfs_raw.items():
    print(f"  {k}: {len(df)} 行")

# 按 Dynamics_of_dominance 的思路：先尽量用两视角补全第三视角的 2D
dfs = fill_missing_2d_from_two_views(cal, dfs_raw)
print("Step1: 已用另两视角补全单视角缺失的 2D 点（fill_missing_2d_from_two_views）。")

points_by_frame = get_points_per_frame(dfs, use_fish_id=False)
frames = sorted(points_by_frame.keys())
print(f"Step1: 共 {len(frames)} 帧。前 5 帧各相机点数：")
for f in frames[:5]:
    cams = points_by_frame[f]
    desc = ", ".join(f"{cam}={len(pts)}" for cam, pts in cams.items())
    print(f"  frame {f}: {desc}")

cam_counts = Counter(len(points_by_frame[f]["cam0"]) for f in frames)
print("Step1: cam0 每帧点数分布 (点数→帧数):", dict(cam_counts))

Step1: 原始 processed CSV 加载完成（行数）:
  player1: 2737 行
  player2: 2655 行
  player3: 2724 行
Step1: 已用另两视角补全单视角缺失的 2D 点（fill_missing_2d_from_two_views）。
Step1: 共 929 帧。前 5 帧各相机点数：
  frame 0: cam0=3, cam1=3, cam2=2
  frame 1: cam0=3, cam1=3, cam2=2
  frame 2: cam0=3, cam1=3, cam2=3
  frame 3: cam0=3, cam1=3, cam2=3
  frame 4: cam0=3, cam1=3, cam2=3
Step1: cam0 每帧点数分布 (点数→帧数): {3: 879, 2: 50}


In [3]:
# Step 2：对每帧计算所有 cam0×cam1×cam2 组合的 3D 候选及配准误差
from scripts.reconstruction.step2_compute_3d_candidates import compute_candidates

candidates_by_frame = compute_candidates(
    cal,
    points_by_frame,
    xyz_bounds=XYZ_BOUNDS,
    fill_missing_view=True,
)
print(f"Step2: 共 {len(candidates_by_frame)} 帧候选。")

num_cands = [len(v) for v in candidates_by_frame.values()]
print(
    f"  每帧候选数: min={min(num_cands)}, max={max(num_cands)}, "
    f"mean={sum(num_cands) / len(num_cands):.1f}"
)

# 按 cost 从小到大排序，优先看几何最自洽的组合
first_frame = frames[0]
cs = candidates_by_frame[first_frame]
cs_sorted = sorted(
    cs,
    key=lambda x: float("inf") if x["cost"] is None or (isinstance(x["cost"], float) and x["cost"] != x["cost"]) else x["cost"],
)
print(f"Step2: frame {first_frame} 的 cost 最小的前 5 个候选 (cam0_idx, cam1_idx, cam2_idx, cost):")
for c in cs_sorted[:5]:
    print(
        "  (cam0=%d, cam1=%d, cam2=%d), cost=%.3f"
        % (c["cam0_idx"], c["cam1_idx"], c["cam2_idx"], c["cost"])
    )

Step2 3D候选: 100%|██████████| 929/929 [02:28<00:00,  6.28frame/s]

Step2: 共 929 帧候选。
  每帧候选数: min=6, max=27, mean=24.5
Step2: frame 0 的 cost 最小的前 5 个候选 (cam0_idx, cam1_idx, cam2_idx, cost):
  (cam0=0, cam1=2, cam2=1), cost=562.614
  (cam0=1, cam1=0, cam2=1), cost=1639.860
  (cam0=0, cam1=1, cam2=1), cost=1706.187
  (cam0=1, cam1=1, cam2=1), cost=2004.332
  (cam0=2, cam1=1, cam2=1), cost=2179.137


In [14]:
mid_frame = 0
print(f"Step2: 示例 frame {mid_frame} 的前 5 个候选 (idx, cost):")
for c in candidates_by_frame[mid_frame][:20]:
    print(
        "  (cam0=%d, cam1=%d, cam2=%d), cost=%.3f"
        % (c["cam0_idx"], c["cam1_idx"], c["cam2_idx"], c["cost"])
    )

Step2: 示例 frame 0 的前 5 个候选 (idx, cost):
  (cam0=0, cam1=0, cam2=0), cost=6533.096
  (cam0=0, cam1=0, cam2=1), cost=2416.538
  (cam0=0, cam1=1, cam2=0), cost=4781.656
  (cam0=0, cam1=1, cam2=1), cost=1706.187
  (cam0=0, cam1=2, cam2=0), cost=2527.369
  (cam0=0, cam1=2, cam2=1), cost=562.614
  (cam0=1, cam1=0, cam2=0), cost=6192.371
  (cam0=1, cam1=0, cam2=1), cost=1639.860
  (cam0=1, cam1=1, cam2=0), cost=5515.754
  (cam0=1, cam1=1, cam2=1), cost=2004.332
  (cam0=1, cam1=2, cam2=0), cost=5487.415
  (cam0=1, cam1=2, cam2=1), cost=3086.707
  (cam0=2, cam1=0, cam2=0), cost=7345.013
  (cam0=2, cam1=0, cam2=1), cost=2390.787
  (cam0=2, cam1=1, cam2=0), cost=6092.275
  (cam0=2, cam1=1, cam2=1), cost=2179.137
  (cam0=2, cam1=2, cam2=0), cost=5840.586
  (cam0=2, cam1=2, cam2=1), cost=3038.162


In [ ]:
# Step 3：在候选中为每帧选择 a/b/c 三条 3D 轨迹（或更少）
from scripts.reconstruction.step3_select_best_correspondence import select_best_all_frames

best = select_best_all_frames(candidates_by_frame, xyz_bounds=XYZ_BOUNDS)
print(f"Step3: 有 {len(best)} 帧成功选出至少 1 条 3D 轨迹。")

from collections import Counter
cnt_groups = Counter(len(info.get("groups", {})) for info in best.values())
print("  每帧 3D 个体数分布 (个体数→帧数):", dict(cnt_groups))

print("Step3: 前 5 帧的 groups（若存在）:")
for f in frames[:5]:
    g = best.get(f, {}).get("groups", {})
    print(f"  frame {f}: {g}")

In [ ]:
# Step 4：跨帧 3D 追踪（匈牙利），保证同一物理个体时间上 ID 一致
from scripts.reconstruction.step4_cross_frame_tracking import apply_cross_frame_tracking

best_tracked = apply_cross_frame_tracking(best, flow_weight=5.0)
print(f"Step4: 跨帧追踪完成，帧数 {len(best_tracked)}。")

cnt_tracked = Counter(len(info.get("groups", {})) for info in best_tracked.values())
print("  跨帧追踪后，每帧 3D 个体数分布:", dict(cnt_tracked))

print("Step4: 前 5 帧跨帧追踪后的 groups:")
for f in frames[:5]:
    g = best_tracked.get(f, {}).get("groups", {})
    print(f"  frame {f}: {g}")

In [ ]:
# Step 5：将跨帧追踪后的 3D 结果导出为 CSV，并在时间轴上补全缺帧
from scripts.reconstruction.export_3d_groups import best_by_frame_to_dfs, dfs_to_single_csv
from scripts.reconstruction.fill_missing_3d import ensure_all_frames_and_fill_spline

# 从 best_tracked 构建 a/b/c 三条 3D 轨迹的 DataFrame
dfs_3d = best_by_frame_to_dfs(best_tracked, shift_to_positive=False)

all_frames = frames
print(f"Step5: 轨迹原始帧范围 = [{min(all_frames)}, {max(all_frames)}]，共 {len(all_frames)} 帧。")

# 在时间轴上补全所有缺失帧，再导出为单 CSV
dfs_3d_filled = ensure_all_frames_and_fill_spline(dfs_3d, all_frames)
single_csv_path = dfs_to_single_csv(dfs_3d_filled, SET_KEY, OUTPUT_DIR)
print("Step5: 已导出单 CSV:", single_csv_path)

# 简单检查导出的 CSV 是否每帧都有 3 条记录
import pandas as pd
_df = pd.read_csv(single_csv_path)
_n_per_frame = _df.groupby("frame").size()
_bad = _n_per_frame[_n_per_frame != 3]
print(f"Step5: 导出 CSV 总帧数={len(_n_per_frame)}，每帧 3 个体的帧数={(_n_per_frame == 3).sum()}")
if len(_bad):
    print("  警告：仍有非 3 个体的帧数=", len(_bad), "示例帧:", _bad.index.tolist()[:10])
else:
    print("  导出 CSV 已保证每帧 3 个体。")

### 3D 重建后立即生成 2×2 视频（调试用）

上面运行完 `run_multiview_id_pipeline` 后，已经写出了当前参数下的 3D 结果（`*_3d.csv`）。

本节先立刻生成一版 2×2 视频，用**刚重建出来的 3D 轨迹**做 2D 叠加，方便和后续（例如 Step 6 清洗后）版本对比，定位问题出在哪一步。

In [ ]:
# 3D 重建后立即生成一版 2×2 视频（仅依赖刚导出的 3D 结果）
import re
from pathlib import Path

from scripts.reconstruction.render_3d_plus_views_video import render_2x2_video

try:
    _ = SET_KEY, OUTPUT_DIR, CALIB_MODEL_DIR, XYZ_BOUNDS
except NameError:
    raise RuntimeError("请先运行上面的路径/参数 cell 和 3D pipeline cell。") from None

# === 三视角原视频路径（与下方 cell 相同逻辑，只是这里命名为 early 版本）===
_video_base = Path(r"D:\behavior_clips\20251225_K_1_2_FCMC_clips")
_m = re.search(r"(\d+\.\d+-\d+\.\d+)", SET_KEY)
_time_part = _m.group(1) if _m else ""
_tag = re.search(r"m\d?tf", SET_KEY)
_tag = _tag.group(0) if _tag else "m2tf"

VIDEO_TOP   = _video_base / f"20251225_K_1_2_FCMC_PLAYER1_{_tag}_Courtship_{_time_part}.mp4"
VIDEO_SIDE1 = _video_base / f"20251225_K_1_2_FCMC_PLAYER2_{_tag}_Courtship_{_time_part}.mp4"
VIDEO_SIDE2 = _video_base / f"20251225_K_1_2_FCMC_PLAYER3_{_tag}_Courtship_{_time_part}.mp4"

video_paths = None
if VIDEO_TOP.exists() and VIDEO_SIDE1.exists() and VIDEO_SIDE2.exists():
    video_paths = {"top": VIDEO_TOP.resolve(), "side1": VIDEO_SIDE1.resolve(), "side2": VIDEO_SIDE2.resolve()}

if video_paths is None:
    print("未配置三路视频，已跳过生成。请设置 VIDEO_TOP/SIDE1/SIDE2 或修改 _video_base 后重试。")
else:
    out_mp4 = render_2x2_video(
        output_dir=OUTPUT_DIR,
        set_key=SET_KEY,
        video_paths=video_paths,
        calibration_folder=CALIB_MODEL_DIR,
        fps=166,
        trail_seconds=0.5,
        data_fps=166,
        xyz_lim=tuple(XYZ_BOUNDS),
        cell_height=360,
        out_filename=f"{SET_KEY}_3d_plus_views_2x2_early.mp4",
    )
    print("已生成【3D 重建后】2×2 视频:", out_mp4)

开始写入 2×2 视频，共 929 帧...
写入 2×2 视频: 929/929 (100.0%)
已生成【3D 重建后】2×2 视频: D:\medaka-mating-output\3d_id\20251225_K_1_2_clips_20251225_K_1_2_FCMC_m1tf_Courtship_290.729-296.331_3d_plus_views_2x2_early.mp4


In [ ]:
# （可选）Step 6：3D 轨迹异常值去除与插值补全，使轨迹更平滑、无突变
# 输入为 OUTPUT_DIR 下 {set_key}_3d.csv；输出可覆盖为 _3d.csv 或另存为 _3d_cleaned.csv
from pathlib import Path
from scripts.reconstruction.step6_3d_outlier_interp import process_set_single_csv

# 若使用单 CSV 格式（frame, fish_id, X, Y, Z）
single_csv = Path(OUTPUT_DIR) / f"{SET_KEY}_3d.csv"
if single_csv.exists():
    process_set_single_csv(
        Path(OUTPUT_DIR),
        SET_KEY,
        max_mm_per_frame=10.0,
        in_place=True,
        max_iterations=10,
        interp_method="cubic",
    )
    print("Step 6 已完成（单 CSV 格式）")
else:
    print("未找到单 CSV，跳过 Step 6；若使用 a/b/c 分文件格式可用 step6.process_set()")

**说明**：下面单元格首次运行可能较慢（10–30 秒），主要是在加载 `render_3d_plus_views_video` 时触发的 matplotlib 等依赖，不是卡在 `import re`。若经常运行，可先单独运行下面一个「仅导入」的单元格，再运行本格。

In [ ]:
# 可选：先运行本格做「导入预热」，再运行下面视频格会快很多
from scripts.reconstruction.render_3d_plus_views_video import render_2x2_video
print("render_2x2_video 已加载")

In [ ]:
# 校验：确认每帧都有 3 个个体且无缺失值
import pandas as pd
from pathlib import Path

path = Path(OUTPUT_DIR) / f"{SET_KEY}_3d.csv"
if not path.exists():
    path = Path(OUTPUT_DIR) / f"{SET_KEY}_a_3d.csv"
    if path.exists():
        dfs = {lb: pd.read_csv(Path(OUTPUT_DIR) / f"{SET_KEY}_{lb}_3d.csv") for lb in ["a", "b", "c"]}
        all_frames = sorted(set().union(*[set(d["frame"]) for d in dfs.values()]))
        n_fish_per_frame = [sum(1 for d in dfs.values() if f in d["frame"].values) for f in all_frames]
        missing = [f for f, n in zip(all_frames, n_fish_per_frame) if n != 3]
        print(f"总帧数: {len(all_frames)}，每帧 3 个体的帧数: {sum(1 for n in n_fish_per_frame if n == 3)}")
        if missing:
            print(f"缺个体帧数: {len(missing)}，示例帧: {missing[:5]}")
        else:
            print("每帧均有 3 个个体。")
    else:
        print("未找到 3D CSV，请先运行 Cell 3。")
else:
    df = pd.read_csv(path)
    frames = df["frame"].unique()
    n_per_frame = df.groupby("frame").size()
    bad = n_per_frame[n_per_frame != 3]
    has_nan = df[["X", "Y", "Z"]].isna().any(axis=1).any()
    print(f"总帧数: {len(frames)}，每帧 3 个体的帧数: {(n_per_frame == 3).sum()}")
    if len(bad):
        print(f"非 3 个体的帧数: {len(bad)}，示例: {bad.index.tolist()[:5]}")
    if has_nan:
        print("存在 NaN 坐标。")
    else:
        print("无缺失值（无 NaN）。")
    if (n_per_frame == 3).all() and not has_nan:
        print("校验通过：每帧都有 3 个个体且没有缺失值。")

## 2×2 合成视频（3D 轨迹 + 三视角 2D 完全由 3D 投影）

生成 2 行 2 列共 4 个画面的 MP4：
- **左上**：3D 轨迹动画（随时间推进的轨迹尾迹）
- **右上/左下/右下**：三视角原视频 + **完全由 3D 结果投影得到的 2D 点**叠加（a/b/c 标注）

2D 叠加**完全从 3D 结果得到**，与 3D 画面一致：若校准目录含 `XYZ_to_imCoords.joblib` 则用 3D→2D 重投影；否则用正交投影（top=XY，side1=XZ，side2=YZ，世界坐标按 `xyz_lim` 线性映射）。不再使用原始 2D 追踪数据。需先运行 pipeline（Cell 3）得到 3D CSV。

In [ ]:
import re
from pathlib import Path

# 必须先运行第一个 cell（路径与参数），否则 SET_KEY/OUTPUT_DIR/XYZ_BOUNDS 未定义
try:
    _ = SET_KEY, OUTPUT_DIR, XYZ_BOUNDS
except NameError:
    raise RuntimeError("请先运行本 notebook 第一个 cell（路径与参数），再运行本 cell。") from None

from scripts.reconstruction.render_3d_plus_views_video import render_2x2_video

# === 三视角原视频路径（任选一种方式） ===
# 按命名规则 20251225_K_1_2_FCMC_PLAYER1_m2tf_Courtship_781.536-786.602.mp4，从 SET_KEY 取时间段和 m1tf/m2tf
_video_base = Path(r"D:\behavior_clips\20251225_K_1_2_FCMC_clips")
_m = re.search(r"(\d+\.\d+-\d+\.\d+)", SET_KEY)
_time_part = _m.group(1) if _m else ""
_tag = re.search(r"m\d?tf", SET_KEY)
_tag = _tag.group(0) if _tag else "m2tf"
VIDEO_TOP   = _video_base / f"20251225_K_1_2_FCMC_PLAYER1_{_tag}_Courtship_{_time_part}.mp4"
VIDEO_SIDE1 = _video_base / f"20251225_K_1_2_FCMC_PLAYER2_{_tag}_Courtship_{_time_part}.mp4"
VIDEO_SIDE2 = _video_base / f"20251225_K_1_2_FCMC_PLAYER3_{_tag}_Courtship_{_time_part}.mp4"

# 优先用上面三个路径；若任一不存在则用目录扫描（按时间段+PLAYER1/2/3）自动匹配
video_paths = None
if VIDEO_TOP.exists() and VIDEO_SIDE1.exists() and VIDEO_SIDE2.exists():
    video_paths = {"top": VIDEO_TOP.resolve(), "side1": VIDEO_SIDE1.resolve(), "side2": VIDEO_SIDE2.resolve()}
if video_paths is None and _video_base.exists():
    time_part = _time_part
    video_paths = {}
    for p in _video_base.rglob("*.mp4"):
        if time_part not in p.stem:
            continue
        n = p.stem.upper()
        if "PLAYER1" in n:
            video_paths["top"] = p.resolve()
        elif "PLAYER2" in n:
            video_paths["side1"] = p.resolve()
        elif "PLAYER3" in n:
            video_paths["side2"] = p.resolve()
    if len(video_paths) != 3:
        video_paths = None

if video_paths is None:
    print("未配置三路视频，已跳过生成。请设置 VIDEO_TOP/SIDE1/SIDE2 或 VIDEO_DIR 后重新运行本 cell。")
else:
    out_mp4 = render_2x2_video(
        output_dir=OUTPUT_DIR,
        set_key=SET_KEY,
        video_paths=video_paths,
        calibration_folder=CALIB_MODEL_DIR,
        fps=166,
        trail_seconds=0.5,
        data_fps=166,
        xyz_lim=tuple(XYZ_BOUNDS),
        cell_height=360,
    )
    print("已生成 2×2 视频:", out_mp4)